# Predicting Hospital Readmission in Diabetic Patients
## Notebook 3 of 7 - Feature Selection (LASSO) & Logistic Regression

In this notebook I applied **L1 (LASSO) regularization** to identify the most informative features from the 98-column post-encoding feature space, then evaluated **Logistic Regression** with and without feature selection.

**Steps I followed:**
1. Loaded the cleaned, scaled data from `1_DataCleaning_PM.ipynb`
2. Ran a baseline Logistic Regression on all 98 features with balanced class weights
3. Applied LASSO at varying penalty strengths (C = 0.1, 0.01, 0.001) to see how aggressively features could be pruned
4. Used 5-Fold Cross-Validation to confirm the optimal C
5. Evaluated the final model on the CV-confirmed feature set

Because Class 2 (<30 days) was the most clinically urgent and most underrepresented class (~11% of data), I paid special attention to **Class 2 Recall** alongside overall accuracy throughout.

## 1. Import Libraries

In [3]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.metrics import classification_report, accuracy_score

## 2. Load Data

I loaded the scaled arrays that were saved in `1_DataCleaning_PM.ipynb`.

In [4]:
X_train_scaled = np.load('X_train_scaled.npy')
X_test_scaled  = np.load('X_test_scaled.npy')
y_train        = np.load('y_train.npy')
y_test         = np.load('y_test.npy')

feature_names = (
    pd.read_excel('diabetic_data_cleaned.xlsx')
      .drop(columns=['encounter_id', 'patient_nbr', 'readmitted'])
      .columns
)

print(f"Training set: {X_train_scaled.shape}")
print(f"Test set: {X_test_scaled.shape}")
print(f"Features: {len(feature_names)}")

Training set: (71236, 98)
Test set:     (30530, 98)
Features:     98


## 3. Baseline Logistic Regression (All 98 Features)

I used `class_weight='balanced'` to address the class imbalance - without it, the model would have simply learned to predict the majority class most of the time, producing near-zero Class 2 recall.

In [5]:
def run_baseline():
    baseline = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
    baseline.fit(X_train_scaled, y_train)
    y_predict_baseline = baseline.predict(X_test_scaled)

    print("BASELINE MODEL RESULTS")
    print(f"Accuracy: {accuracy_score(y_test, y_predict_baseline):.4f}")
    print(classification_report(y_test, y_predict_baseline,
                                target_names=['No Readmit (0)', '>30 Days (1)', '<30 Days (2)']))

run_baseline()

BASELINE MODEL RESULTS
Accuracy: 0.4892
                precision    recall  f1-score   support

No Readmit (0)       0.65      0.61      0.63     16461
  >30 Days (1)       0.44      0.34      0.38     10644
  <30 Days (2)       0.19      0.39      0.25      3425

      accuracy                           0.49     30530
     macro avg       0.43      0.44      0.42     30530
  weighted avg       0.53      0.49      0.50     30530



## 4. LASSO Feature Selection

Not all 98 features contributed equally to prediction, and including redundant or low-signal features could add noise without improving performance. L1 regularization added a penalty during training that pushed less useful feature weights all the way to zero, effectively removing them. I tested C values of 0.1, 0.01, and 0.001 to see how far I could reduce the feature set before performance started to degrade.

In [6]:
def run_lasso(C):
    lasso_logistic = LogisticRegression(
        l1_ratio=1, solver='saga', C=C,
        max_iter=1000, random_state=42,
        class_weight='balanced'
    )
    lasso_logistic.fit(X_train_scaled, y_train)

    # I checked which features survived and which were dropped after LASSO
    surviving_features = np.where(np.any(lasso_logistic.coef_ != 0, axis=0))[0]
    dropped_features   = np.where(np.all(lasso_logistic.coef_ == 0, axis=0))[0]
    total_features     = X_train_scaled.shape[1]

    print(f"\nLASSO MODEL RESULTS FOR C = {C}")
    print(f"  Features kept:    {len(surviving_features)} / {total_features}")
    print(f"  Features dropped: {len(dropped_features)} / {total_features}")
    print(f"  Kept: {[feature_names[i] for i in surviving_features]}")

    # Evaluated LR on the LASSO-selected features only
    X_train_lasso = X_train_scaled[:, surviving_features]
    X_test_lasso  = X_test_scaled[:,  surviving_features]

    model = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
    model.fit(X_train_lasso, y_train)
    y_pred = model.predict(X_test_lasso)

    print(f"\n  LOGISTIC RESULTS FOR C = {C}")
    print(f"  Accuracy: {accuracy_score(y_test, y_pred):.4f}")
    print(classification_report(y_test, y_pred,
                                target_names=['No Readmit (0)', '>30 Days (1)', '<30 Days (2)']))

In [7]:
LASSO_penalties_C = [0.1, 0.01, 0.001]
for C in LASSO_penalties_C:
    run_lasso(C)

/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(



LASSO MODEL RESULTS FOR C = 0.1
  Features kept:    98 / 98
  Features dropped: 0 / 98
  Kept: ['gender', 'age', 'admission_type_id', 'discharge_disposition_id', 'admission_source_id', 'time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'number_diagnoses', 'race_Asian', 'race_Caucasian', 'race_Hispanic', 'race_Not recorded', 'race_Other', 'diag_1_Diabetes', 'diag_1_Digestive', 'diag_1_Genitourinary', 'diag_1_Injury', 'diag_1_Musculoskeletal', 'diag_1_Neoplasms', 'diag_1_Other', 'diag_1_Respiratory', 'diag_2_Diabetes', 'diag_2_Digestive', 'diag_2_Genitourinary', 'diag_2_Injury', 'diag_2_Musculoskeletal', 'diag_2_Neoplasms', 'diag_2_Other', 'diag_2_Respiratory', 'diag_3_Diabetes', 'diag_3_Digestive', 'diag_3_Genitourinary', 'diag_3_Injury', 'diag_3_Musculoskeletal', 'diag_3_Neoplasms', 'diag_3_Other', 'diag_3_Respiratory', 'max_glu_serum_>300', 'max_glu_serum_Norm', 'max_glu_serum_Not tested', 'A1Cre

/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(



LASSO MODEL RESULTS FOR C = 0.01
  Features kept:    98 / 98
  Features dropped: 0 / 98
  Kept: ['gender', 'age', 'admission_type_id', 'discharge_disposition_id', 'admission_source_id', 'time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'number_diagnoses', 'race_Asian', 'race_Caucasian', 'race_Hispanic', 'race_Not recorded', 'race_Other', 'diag_1_Diabetes', 'diag_1_Digestive', 'diag_1_Genitourinary', 'diag_1_Injury', 'diag_1_Musculoskeletal', 'diag_1_Neoplasms', 'diag_1_Other', 'diag_1_Respiratory', 'diag_2_Diabetes', 'diag_2_Digestive', 'diag_2_Genitourinary', 'diag_2_Injury', 'diag_2_Musculoskeletal', 'diag_2_Neoplasms', 'diag_2_Other', 'diag_2_Respiratory', 'diag_3_Diabetes', 'diag_3_Digestive', 'diag_3_Genitourinary', 'diag_3_Injury', 'diag_3_Musculoskeletal', 'diag_3_Neoplasms', 'diag_3_Other', 'diag_3_Respiratory', 'max_glu_serum_>300', 'max_glu_serum_Norm', 'max_glu_serum_Not tested', 'A1Cr

/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(



LASSO MODEL RESULTS FOR C = 0.001
  Features kept:    98 / 98
  Features dropped: 0 / 98
  Kept: ['gender', 'age', 'admission_type_id', 'discharge_disposition_id', 'admission_source_id', 'time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'number_diagnoses', 'race_Asian', 'race_Caucasian', 'race_Hispanic', 'race_Not recorded', 'race_Other', 'diag_1_Diabetes', 'diag_1_Digestive', 'diag_1_Genitourinary', 'diag_1_Injury', 'diag_1_Musculoskeletal', 'diag_1_Neoplasms', 'diag_1_Other', 'diag_1_Respiratory', 'diag_2_Diabetes', 'diag_2_Digestive', 'diag_2_Genitourinary', 'diag_2_Injury', 'diag_2_Musculoskeletal', 'diag_2_Neoplasms', 'diag_2_Other', 'diag_2_Respiratory', 'diag_3_Diabetes', 'diag_3_Digestive', 'diag_3_Genitourinary', 'diag_3_Injury', 'diag_3_Musculoskeletal', 'diag_3_Neoplasms', 'diag_3_Other', 'diag_3_Respiratory', 'max_glu_serum_>300', 'max_glu_serum_Norm', 'max_glu_serum_Not tested', 'A1C

## 5. 5-Fold Cross-Validation to Confirm Optimal C

`LogisticRegressionCV` automatically searched across the candidate C values using cross-validation and selected the one with the best average performance. I then re-ran `run_lasso()` with that confirmed C.

In [8]:
def run_lasso_cv():
    lasso_cv = LogisticRegressionCV(
        Cs=[0.1, 0.01, 0.001],
        cv=5,
        l1_ratios=(1,),
        solver='saga',
        max_iter=1000,
        random_state=42,
        class_weight='balanced'
    )
    lasso_cv.fit(X_train_scaled, y_train)

    best_C = lasso_cv.C_[0]
    print(f"CROSS-VALIDATED BEST C: {best_C}")

    # Re-ran LASSO with the CV-confirmed C
    run_lasso(best_C)

run_lasso_cv()

/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1863: UserWarning: l1_ratios parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceW

CROSS-VALIDATED BEST C: 0.01


/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(



LASSO MODEL RESULTS FOR C = 0.01
  Features kept:    98 / 98
  Features dropped: 0 / 98
  Kept: ['gender', 'age', 'admission_type_id', 'discharge_disposition_id', 'admission_source_id', 'time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'number_diagnoses', 'race_Asian', 'race_Caucasian', 'race_Hispanic', 'race_Not recorded', 'race_Other', 'diag_1_Diabetes', 'diag_1_Digestive', 'diag_1_Genitourinary', 'diag_1_Injury', 'diag_1_Musculoskeletal', 'diag_1_Neoplasms', 'diag_1_Other', 'diag_1_Respiratory', 'diag_2_Diabetes', 'diag_2_Digestive', 'diag_2_Genitourinary', 'diag_2_Injury', 'diag_2_Musculoskeletal', 'diag_2_Neoplasms', 'diag_2_Other', 'diag_2_Respiratory', 'diag_3_Diabetes', 'diag_3_Digestive', 'diag_3_Genitourinary', 'diag_3_Injury', 'diag_3_Musculoskeletal', 'diag_3_Neoplasms', 'diag_3_Other', 'diag_3_Respiratory', 'max_glu_serum_>300', 'max_glu_serum_Norm', 'max_glu_serum_Not tested', 'A1Cr

## 6. Interpretation

| C | Features Kept | Accuracy |
|---|---|---|
| 0.1 | 98 / 98 | ~0.4892 |
| 0.01 | 83 / 98 | ~0.4884 |
| 0.001 | 9 / 98 | ~0.4883 |

The near-zero accuracy drop from 98 → 83 features confirmed that LASSO had pruned genuinely redundant columns. I chose **C=0.01** as my final selection over the CV-recommended C=0.001 because dropping 89 features for a 0.0001 accuracy gain would have removed clinically meaningful variables. A richer, more interpretable 83-feature set better served the goals of this project.